In [0]:
EH_CONN_STR="CONN_STR"
EH_NAME="orders"
EH_NAMESPACE="eh-ns-dbxxprj"

In [0]:
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": f"{EH_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EH_NAME,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";',
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "30000",
    "maxOffsetsPerTrigger": "50000",
    "failOnDataLoss": "true",
    "startingOffsets": "earliest",
}

In [0]:
checkpoint_dir = "/Volumes/adb_dbxproject/00_landing/checkpoints/eventHub"

# 1. Clean up the folder first if starting fresh
dbutils.fs.rm(checkpoint_dir, recurse=True)

# 2. Start persistent streaming ingestion
query = (
    df_raw.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True) # or .trigger(processingTime="10 seconds")
    .toTable("adb_dbxproject.00_landing.eventhub_raw")
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("restaurant_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_type", StringType(), True),
    StructField("items", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True)
])

In [0]:
df = spark.read.format("delta").table("adb_dbxproject.00_landing.eventhub_raw")

df_parsed = (
    df
    .withColumn("key_str", col("key").cast("string"))
    .withColumn("value_str", col("value").cast("string"))
    .withColumn("data", from_json(col("value_str"),order_schema))
    .select("data.*")
    .withColumnRenamed("timestamp", "order_timestamp")
)

display(df_parsed)